Colab is making it easier than ever to integrate powerful Generative AI capabilities into your projects. We are launching public preview for a simple and intuitive Python library (google.colab.ai) to access state-of-the-art language models directly within Colab environments. All users have free access to most popular LLMs, while paid users have access to a wider selection of models. This means users can spend less time on configuration and set up and more time bringing their ideas to life. With just a few lines of code, you can now perform a variety of tasks:
- Generate text
- Translate languages
- Write creative content
- Categorize text

Happy Coding!


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb)

In [ ]:
# @title List available models
from google.colab import ai

ai.list_models()

['google/gemini-2.0-flash',
 'google/gemini-2.0-flash-lite',
 'google/gemini-2.5-flash',
 'google/gemini-2.5-flash-lite',
 'google/gemini-2.5-pro',
 'google/gemma-3-12b',
 'google/gemma-3-1b',
 'google/gemma-3-27b',
 'google/gemma-3-4b']

Choosing a Model
The model names give you a hint about their capabilities and intended use:

Pro: These are the most capable models, ideal for complex reasoning, creative tasks, and detailed analysis.

Flash: These models are optimized for high speed and efficiency, making them great for summarization, chat applications, and tasks requiring rapid responses.

Gemma: These are lightweight, open-weight models suitable for a variety of text generation tasks and are great for experimentation.

In [ ]:
# @title Simple batch generation example
# Only text-to-text input/output is supported
from google.colab import ai

response = ai.generate_text("What is the capital of France?")
print(response)

The capital of France is **Paris**.



In [ ]:
# @title Choose a different model
from google.colab import ai

response = ai.generate_text("What is the capital of England", model_name='google/gemini-2.0-flash-lite')
print(response)

The capital of England is **London**.



For longer text generations, you can stream the response. This displays the output token by token as it's generated, rather than waiting for the entire response to complete. This provides a more interactive and responsive experience. To enable this, simply set stream=True.

In [ ]:
# @title Simple streaming example
from google.colab import ai

stream = ai.generate_text("Tell me a short story.", stream=True)
for text in stream:
  print(text, end='')

The lighthouse keeper, Silas, was a man of routine. Every night, for fifty years, he'd lit the lamp, a beacon against the treacherous rocks that gnawed at the coastline. The sea was his companion, his enemy, and his only confidante. He knew its moods better than his own.

One stormy night, the wind howled like a banshee. The waves crashed against the tower, shaking it to its core. Silas, clinging to the railing, felt a fear he hadn't experienced in decades. This wasn't just a storm; this was a monster.

Suddenly, a small, wooden boat, tossed about like a toy, appeared in the raging sea. He squinted, his heart leaping into his throat. A child. Alone.

Ignoring the raging tempest, Silas raced down the winding stairs, his old bones protesting with every step. He launched his small rescue boat, a fragile craft against the fury of the storm.

Fighting the waves, he reached the child. A girl, no older than seven, clung to the wreckage, her face white with terror. With a strength born of desp

In [ ]:
#@title Text formatting setup
#code is not necessary for colab.ai, but is useful in fomatting text chunks
import sys

class LineWrapper:
    def __init__(self, max_length=80):
        self.max_length = max_length
        self.current_line_length = 0

    def print(self, text_chunk):
        i = 0
        n = len(text_chunk)
        while i < n:
            start_index = i
            while i < n and text_chunk[i] not in ' \n': # Find end of word
                i += 1
            current_word = text_chunk[start_index:i]

            delimiter = ""
            if i < n: # If not end of chunk, we found a delimiter
                delimiter = text_chunk[i]
                i += 1 # Consume delimiter

            if current_word:
                needs_leading_space = (self.current_line_length > 0)

                # Case 1: Word itself is too long for a line (must be broken)
                if len(current_word) > self.max_length:
                    if needs_leading_space: # Newline if current line has content
                        sys.stdout.write('\n')
                        self.current_line_length = 0
                    for char_val in current_word: # Break the long word
                        if self.current_line_length >= self.max_length:
                            sys.stdout.write('\n')
                            self.current_line_length = 0
                        sys.stdout.write(char_val)
                        self.current_line_length += 1
                # Case 2: Word doesn't fit on current line (print on new line)
                elif self.current_line_length + (1 if needs_leading_space else 0) + len(current_word) > self.max_length:
                    sys.stdout.write('\n')
                    sys.stdout.write(current_word)
                    self.current_line_length = len(current_word)
                # Case 3: Word fits on current line
                else:
                    if needs_leading_space:
                        # Define punctuation that should not have a leading space
                        # when they form an entire "word" (token) following another word.
                        no_leading_space_punctuation = {
                            ",", ".", ";", ":", "!", "?",        # Standard sentence punctuation
                            ")", "]", "}",                     # Closing brackets
                            "'s", "'S", "'re", "'RE", "'ve", "'VE", # Common contractions
                            "'m", "'M", "'ll", "'LL", "'d", "'D",
                            "n't", "N'T",
                            "...", "…"                          # Ellipses
                        }
                        if current_word not in no_leading_space_punctuation:
                            sys.stdout.write(' ')
                            self.current_line_length += 1
                    sys.stdout.write(current_word)
                    self.current_line_length += len(current_word)

            if delimiter == '\n':
                sys.stdout.write('\n')
                self.current_line_length = 0
            elif delimiter == ' ':
                # If line is full and a space delimiter arrives, it implies a wrap.
                if self.current_line_length >= self.max_length:
                    sys.stdout.write('\n')
                    self.current_line_length = 0

        sys.stdout.flush()


In [ ]:
# @title Formatted streaming example
from google.colab import ai

wrapper = LineWrapper()
for chunk in ai.generate_text('Give me a long winded description about the evolution of the Roman Empire.', model_name='google/gemini-2.0-flash', stream=True):
  wrapper.print(chunk)

Alright, settle in, because the Roman Empire’s evolution wasn't a tidy, linear
process. It was a centuries-long, tumultuous transformation, marked by
breathtaking innovation, brutal power struggles, and a slow, creeping societal
decay. We're talking about a journey from a humble city-state in the Italian
peninsula to a sprawling, multifaceted empire that left an indelible mark on
law, language, architecture, governance, and even our very understanding of the
world.

It all began, as legend would have it, with Romulus and Remus, twin brothers
raised by a she-wolf, who founded the city of Rome in 753 BCE. Now, that’s just
a legend, but it serves to highlight the foundational spirit of Rome: ambition,
strength, and a certain ruthlessness. Initially, Rome was ruled by a monarchy, a
system eventually deemed unsatisfactory by the powerful patrician class. This
led to the **Roman Republic**, established around 509 BCE, a watershed moment
that would define the early character of Rome.

The Rep

In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Tribal Knowledge Assistant</title>
    <!-- Inter font -->
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
    <!-- MSAL.js for Azure Entra ID authentication -->
    <script src="https://alcdn.msauth.net/browser/2.38.0/js/msal-browser.min.js"></script>
    <!-- Marked.js for Markdown parsing -->
    <script src="https://cdn.jsdelivr.net/npm/marked@9.1.6/marked.min.js"></script>
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', 'Roboto', sans-serif;
            background: linear-gradient(135deg, #0f172a 0%, #1e293b 50%, #334155 100%);
            color: #e2e8f0;
            height: 100vh;
            display: flex;
            flex-direction: column;

        }
        .home-container {
            display: flex;
            flex-direction: column, row;
            align-items: center;
            justify-content: center;
            box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);
            min-height: 100%; /* Use min-height instead of height */
            margin: 0;
        }


        .home-card {
            text-align: center;
            width: 100%;
            max-width: 90%;
            box-sizing: border-box;
            box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);
        }

        .home-card h1 {
            font-size: 1.8;
        }


        .common-questions {
            display: flex !important;
            flex-direction: row !important;
            flex-wrap: wrap;
            gap: 20px;
            justify-content: flex-start;
            margin-top: 30px;
        }

        .common-questions > .common-question,
        .common-questions > .list-group-item,
        .common-questions > li {
            flex: 0 1 calc((100% - 40px) / 3);
            max-width: calc((100% - 40px) / 3);
            width: auto !important;
            margin: 0;
        }

        .common-questions .list-group-item-action {
            width: auto !important;
            display: block;
        }

        .common-question {
    padding: 19px;
    font-size: 1.1rem;
    background: linear-gradient(135deg, #1757a5, #3b82f6);
    border-radius: 16px; /* More oval shape */
    box-shadow: 0 2px 6px rgba(0,0,0,0.1);
    color: #f3f2f2;
    cursor: pointer;
    transition: all 0.8s ease;
    display: flex;
    flex-direction: column;
    justify-content: flex-start;
    align-items: flex-start;
    text-align: left;
    margin-bottom: 12px;
}

.common-desc {
    padding-top: 8px;
    font-size: 0.9rem;
    color: #ebebeb;
}

#seeMoreBtn {
    margin-top: 10px;
    background-color: transparent;
    border: none;
    color: #9ab9eb;
    font-size: 1rem;
    cursor: pointer;
}


.see-more-container {
    display: flex;
    justify-content: flex-end;
}



        .common-question:hover {
            transform: translateY(-5px);
            box-shadow: 0 6px 10px rgba(2, 9, 42, 0.2);
        }

        .common-question:active {
            transform: translateY(0);
            box-shadow: 0 4px 6px rgba(25, 1, 45, 0.1);
        }

        /* Login screen styles */
        .login-container {
            display: flex;
            align-items: center;
            justify-content: center;
            height: 100vh;
            background: linear-gradient(135deg, #0f172a 0%, #1e293b 50%, #334155 100%);
        }

        .login-card {
            background: rgba(15, 23, 42, 0.95);
            backdrop-filter: blur(20px);
            padding: 3rem;
            border-radius: 1.5rem;
            border: 1px solid rgba(255, 255, 255, 0.2);
            text-align: center;
            max-width: 450px;
            width: 90%;
            box-shadow: 0 25px 50px -12px rgba(0, 0, 0, 0.4);
        }

        .login-card h1 {
            margin-bottom: 1.5rem;
            background: linear-gradient(135deg, #60a5fa, #3b82f6);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            background-clip: text;
            font-weight: 700;
            font-size: 1.75rem;
        }

        .login-button {
            background: linear-gradient(135deg, #3b82f6 0%, #1d4ed8 100%);
            color: white;
            border: none;
            padding: 1rem 2.5rem;
            border-radius: 0.75rem;
            font-size: 1rem;
            font-weight: 600;
            cursor: pointer;
            display: inline-flex;
            align-items: center;
            gap: 0.75rem;
            transition: all 0.3s ease;
            box-shadow: 0 4px 15px rgba(59, 130, 246, 0.4);
        }

        .login-button:hover {
            background: linear-gradient(135deg, #2563eb 0%, #1e40af 100%);
            transform: translateY(-2px);
            box-shadow: 0 8px 25px rgba(59, 130, 246, 0.6);
        }

        .login-button:disabled {
            background: #94a3b8;
            cursor: not-allowed;
            transform: none;
            box-shadow: none;
        }

        /* Chat interface styles (existing) */
        .header {
            background: rgba(15, 23, 42, 0.95);
            backdrop-filter: blur(20px);
            padding: 1rem 2rem;
            border-bottom: 1px solid rgba(255, 255, 255, 0.1);
            display: flex;
            justify-content: space-between;
            align-items: center;
            position: relative;
            box-shadow: 0 1px 3px rgba(0, 0, 0, 0.3);
        }

        .header-left {
            display: flex;
            align-items: center;
            gap: 1rem;
        }

        .header h1 {
            font-size: 1.5rem;
            font-weight: 700;
            background: linear-gradient(135deg, #60a5fa, #3b82f6);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            background-clip: text;
        }

        .new-chat-button {
            background: rgba(255, 255, 255, 0.1);
            border: 1px solid rgba(255, 255, 255, 0.2);
            border-radius: 0.75rem;
            color: #e2e8f0;
            padding: 0.75rem 1.25rem;
            font-size: 0.875rem;
            font-weight: 500;
            cursor: pointer;
            display: flex;
            align-items: center;
            gap: 0.75rem;
            transition: all 0.3s ease;
            backdrop-filter: blur(10px);
        }

        .new-chat-button:hover {
            background: rgba(255, 255, 255, 0.15);
            border-color: rgba(255, 255, 255, 0.3);
            transform: translateY(-1px);
            box-shadow: 0 4px 12px rgba(59, 130, 246, 0.2);
        }

        .sidebar-toggle {
            background: rgba(255, 255, 255, 0.1);
            border: 1px solid rgba(255, 255, 255, 0.2);
            border-radius: 0.75rem;
            color: #e2e8f0;
            padding: 0.75rem;
            cursor: pointer;
            display: flex;
            align-items: center;
            justify-content: center;
            transition: all 0.3s ease;
            position: relative;
            width: 44px;
            height: 44px;
            backdrop-filter: blur(10px);
        }

        .sidebar-toggle:hover {
            background: rgba(255, 255, 255, 0.15);
            border-color: rgba(255, 255, 255, 0.3);
            transform: translateY(-1px);
            box-shadow: 0 4px 12px rgba(59, 130, 246, 0.2);
        }

        /* Animated sidebar icon */
        .sidebar-icon {
            width: 20px;
            height: 16px;
            position: relative;
            transition: all 0.3s ease;
        }

        .sidebar-icon::before,
        .sidebar-icon::after,
        .sidebar-icon {
            position: relative;
        }

        /* Main panel */
        .sidebar-icon::before {
            content: '';
            position: absolute;
            left: 6px;
            top: 0;
            width: 14px;
            height: 16px;
            background-color: currentColor;
            border-radius: 1px;
            transition: all 0.3s ease;
        }

        /* Side panel */
        .sidebar-icon::after {
            content: '';
            position: absolute;
            left: 0;
            top: 0;
            width: 4px;
            height: 16px;
            background-color: currentColor;
            border-radius: 1px;
            transition: all 0.3s ease;
        }

        /* Collapsed state - transforms into a single panel with arrow */
        .sidebar-toggle.collapsed .sidebar-icon::before {
            left: 0;
            width: 16px;
        }

        .sidebar-toggle.collapsed .sidebar-icon::after {
            left: 18px;
            width: 2px;
            height: 10px;
            top: 3px;
            transform: rotate(0deg);
        }

        /* Add arrow indicator for collapsed state */
        .sidebar-toggle.collapsed .sidebar-icon {
            position: relative;
        }

        .sidebar-toggle.collapsed .sidebar-icon::after {
            content: '';
            position: absolute;
            right: -2px;
            top: 50%;
            transform: translateY(-50%);
            width: 0;
            height: 0;
            border-left: 4px solid currentColor;
            border-top: 3px solid transparent;
            border-bottom: 3px solid transparent;
            background: none;
            border-radius: 0;
        }

        .main-layout {
            display: flex;
            height: calc(100vh - 80px);
        }

        .sidebar {
            width: 280px;
            background: rgba(15, 23, 42, 0.98);
            backdrop-filter: blur(20px);
            border-right: 1px solid rgba(255, 255, 255, 0.1);
            display: flex;
            flex-direction: column;
            transition: transform 0.3s ease;
            box-shadow: 2px 0 10px rgba(0, 0, 0, 0.3);
        }
        .sidebar.collapsed {
            transform: translateX(-100%);
        }

        .sidebar-content {
            padding: 1rem;
            display: flex;
            flex-direction: column;
            height: 100%;
        }

        .sidebar-new-chat-button {
            background: rgba(255, 255, 255, 0.1);
            border: 1px solid rgba(255, 255, 255, 0.2);
            border-radius: 0.75rem;
            color: #e2e8f0;
            padding: 0.75rem 1rem;
            font-size: 0.875rem;
            font-weight: 500;
            cursor: pointer;
            display: flex;
            align-items: center;
            gap: 0.75rem;
            transition: all 0.3s ease;
            margin-bottom: 1rem;
            width: 100%;
            backdrop-filter: blur(10px);
        }

        .sidebar-new-chat-button:hover {
            background: rgba(255, 255, 255, 0.15);
            border-color: rgba(255, 255, 255, 0.3);
            transform: translateY(-1px);
        }

        .chat-history {
            flex: 1;
            display: flex;
            flex-direction: column;
            overflow-y: auto; /* this is a must for scroll bars to make sure page doesnt extend badly */
        }

        .chat-history-header {
            margin-bottom: 0.75rem;
        }

        .chat-history-header h3 {
            font-size: 0.875rem;
            font-weight: 600;
            color: #94a3b8;
            margin: 0;
        }

        .chat-list {
            flex: 1;
            overflow-y: auto;
            scroll-behavior: smooth;
        }

        .chat-list {
            /* Modern browsers: thin auto-hiding scrollbar */
            scrollbar-width: thin;
            scrollbar-color: #565869 transparent;
        }

        /* Webkit browsers (Safari/Chrome): custom auto-hiding scrollbar */
        .chat-list::-webkit-scrollbar {
            width: 6px;
        }

        .chat-list::-webkit-scrollbar-track {
            background: transparent;
        }

        .chat-list::-webkit-scrollbar-thumb {
            background-color: #565869;
            border-radius: 3px;
            /* Auto-hide when not hovering/scrolling */
            opacity: 0;
            transition: opacity 0.3s ease;
        }

        .chat-list::-webkit-scrollbar-thumb:hover {
            background-color: #6f7089;
            opacity: 1;
        }

        /* Show scrollbar when container is hovered or being scrolled */
        .chat-list:hover::-webkit-scrollbar-thumb {
            opacity: 0.7;
        }

        .chat-list::-webkit-scrollbar-thumb:active {
            opacity: 1;
        }

        .chat-preview-container {
            position: relative;
        }

        .chat-preview-badge {
            display: flex;
            align-items: center;
            gap: 0.5rem;
            background: linear-gradient(135deg, #3b82f6, #1d4ed8);
            color: white;
            padding: 0.5rem 1rem;
            border-radius: 1rem;
            font-size: 0.75rem;
            font-weight: 600;
            margin-bottom: 1.5rem;
            box-shadow: 0 4px 15px rgba(59, 130, 246, 0.4);
        }

        .chat-preview-badge svg {
            flex-shrink: 0;
        }

        .chat-item {
            padding: 0.75rem;
            margin-bottom: 0.5rem;
            border-radius: 0.5rem;
            cursor: pointer;
            transition: all 0.2s;
            border: 1px solid transparent;
            display: flex;
            align-items: flex-start;
            gap: 0.75rem;
            position: relative;
        }

        .chat-item.preview {
            opacity: 0.8;
            cursor: allowed;
            border: 1px dashed rgba(59, 130, 246, 0.3);
            background: linear-gradient(135deg, rgba(59, 130, 246, 0.05), rgba(29, 78, 216, 0.02));
        }

        .chat-item.preview:hover {
            opacity: 0.9;
            background: linear-gradient(135deg, rgba(59, 130, 246, 0.1), rgba(29, 78, 216, 0.05));
            border-color: rgba(59, 130, 246, 0.4);
        }

        .chat-item:not(.preview):hover {
            background-color: #40414f;
        }

        .chat-item.active {
            background-color: #343541;
            border-color: #565869;
        }

        .chat-item-content {
            flex: 1;
            min-width: 0;
        }

        .chat-item-title {
            font-size: 0.875rem;
            color: #ececf1;
            margin: 0 0 0.375rem 0;
            white-space: nowrap;
            overflow: hidden;
            text-overflow: ellipsis;
            font-weight: 500;
        }

        .chat-item-preview {
            font-size: 0.75rem;
            color: #8e8ea0;
            margin: 0 0 0.5rem 0;
            line-height: 1.4;
            display: -webkit-box;
            line-clamp: 2;
            -webkit-box-orient: vertical;
            overflow: hidden;
        }

        .chat-item-time {
            font-size: 0.6875rem;
            color: #565869;
            margin: 0;
            font-weight: 500;
        }

        .chat-item-indicator {
            display: flex;
            flex-direction: column;
            align-items: center;
            gap: 0.25rem;
        }

        .message-count {
            background-color: #565869;
            color: #ececf1;
            font-size: 0.625rem;
            font-weight: 600;
            padding: 0.25rem 0.5rem;
            border-radius: 0.75rem;
            min-width: 1.25rem;
            text-align: center;
            line-height: 1;
        }

        .chat-item.preview .message-count {
            background-color: rgba(86, 88, 105, 0.5);
            color: rgba(236, 236, 241, 0.6);
        }

        .main-content {
            flex: 1;
            display: flex;
            flex-direction: column;
            min-width: 0; /* Allows flex child to shrink */
            overflow: hidden;
        }

        /* Responsive sidebar */
        @media (max-width: 768px) {
            .sidebar {
                position: fixed;
                top: 80px;
                left: 0;
                height: calc(100vh - 80px);
                z-index: 100;
                box-shadow: 2px 0 10px rgba(0, 0, 0, 0.3);
            }

            .main-content {
                width: 100%;
            }

            /* On mobile, always use collapsed spacing since sidebar overlays */
            .main-layout,
            .main-layout.sidebar-collapsed,
            .main-layout:not(.sidebar-collapsed) {
                --chat-padding: clamp(1rem, 5vw + 1rem, 20vw) !important;
            }
        }

        .header-right {
            display: flex;
            align-items: center;
            gap: 0.5rem;
        }

        .header-pill {
            display: inline-flex;
            align-items: center;
            justify-content: center;
            gap: 0.75rem;
            height: 44px;
            padding: 0.75rem 1.25rem;
            background: rgba(255, 255, 255, 0.1);
            backdrop-filter: blur(10px);
            border-radius: 0.75rem;
            border: 1px solid rgba(255, 255, 255, 0.2);
            color: #e2e8f0;
            font-size: 0.875rem;
            font-weight: 500;
        }

        button.header-pill {
            cursor: pointer;
            transition: all 0.3s ease;
        }

        button.header-pill:hover {
            background: rgba(255, 255, 255, 0.15);
            border-color: rgba(255, 255, 255, 0.3);
            transform: translateY(-1px);
            box-shadow: 0 4px 12px rgba(59, 130, 246, 0.2);
        }

        .user-info {
            display: flex;
            align-items: center;
            gap: 0.75rem;
            font-size: 0.875rem;
        }

        .user-avatar {
            width: 32px;
            height: 32px;
            border-radius: 8px;
            background: linear-gradient(135deg, #3b82f6, #1d4ed8);
            display: flex;
            align-items: center;
            justify-content: center;
            font-weight: 600;
            font-size: 0.75rem;
            color: white;
            box-shadow: 0 2px 8px rgba(59, 130, 246, 0.3);
        }

        .logout-button {
            width: 44px;
            padding: 0;
        }

        .logout-button:hover {
            background: rgba(255, 255, 255, 0.15);
            border-color: rgba(255, 255, 255, 0.3);
            transform: translateY(-1px);
        }

        .status-indicator {
            display: flex;
            align-items: center;
            gap: 0.5rem;
            font-size: 0.87 5rem;
        }

        .status-dot {
            width: 8px;
            height: 8px;
            border-radius: 50%;
            background-color: #10a37f;
        }

        /* New styles for the version button */
        .version-button { /* RENAMED from .version-text */
            margin-left: 0.25rem; /* Adjust spacing between pills */
            font-size: 0.75rem;
            color: #e2e8f0;
            font-weight: 500;
            background: rgba(255, 255, 255, 0.1);
            border: 1px solid rgba(255, 255, 255, 0.2);
            cursor: pointer;
            border-radius: 0.75rem;
            transition: all 0.3s ease;
            display: inline-flex;
            align-items: center;
            justify-content: center;
            height: 44px;
            padding: 0 1.25rem;
        }

        .version-button:hover {
            background: rgba(255, 255, 255, 0.15);
            border-color: rgba(255, 255, 255, 0.3);
            transform: translateY(-1px);
            box-shadow: 0 4px 12px rgba(59, 130, 246, 0.2);
        }

        .chat-container {
            flex: 1;
            display: flex;
            flex-direction: column;
            margin: 0 auto;
            width: 100%;
            height: calc(100vh - 80px);
            position: relative;
            padding: 0 var(--chat-padding);
            transition: padding 0.3s ease;
        }

        /* Dynamic padding based on available width */
        :root {
            --chat-padding: clamp(1rem, 5vw + 1rem, 20vw);
        }

        /* When sidebar is expanded (less space available) - SMALLER margins */
        .main-layout:not(.sidebar-collapsed) {
            --chat-padding: clamp(1rem, 2vw + 0.5rem, 10vw);
        }

        /* When sidebar is collapsed (more space available) - LARGER margins */
        .main-layout.sidebar-collapsed {
            --chat-padding: clamp(1rem, 6vw + 1rem, 25vw);
        }

        /* Only override for very small screens */
        @media (max-width: 480px) {
            .chat-container {
                padding: 0 0.75rem;
            }
        }

        .messages-container {
            flex: 1;
            overflow-y: auto;
            padding: 2rem;
            scroll-behavior: smooth;
            min-height: 0;
        }

        .message {
            margin-bottom: 2rem;
            display: flex;
            gap: 1.25rem;
            align-items: flex-start;
            padding: 1.5rem;
            border-radius: 1rem;
            backdrop-filter: blur(10px);
            border: 1px solid rgba(0, 0, 0, 0.05);
            transition: all 0.3s ease;
        }

        .message.user {
            background: linear-gradient(135deg, rgba(59, 130, 246, 0.2) 0%, rgba(29, 78, 216, 0.1) 100%);
            margin-left: 2rem;
            border: 1px solid rgba(59, 130, 246, 0.2);
        }

        .message.assistant {
            background: linear-gradient(135deg, rgba(15, 23, 42, 0.8) 0%, rgba(30, 41, 59, 0.9) 100%);
            backdrop-filter: blur(20px);
            box-shadow: 0 8px 32px rgba(0, 0, 0, 0.3);
            border: 1px solid rgba(255, 255, 255, 0.1);
        }

        .message:hover {
            transform: translateY(-1px);
            box-shadow: 0 8px 25px rgba(0, 0, 0, 0.1);
        }

        .message-avatar {
            width: 40px;
            height: 40px;
            border-radius: 12px;
            display: flex;
            align-items: center;
            justify-content: center;
            font-weight: 600;
            font-size: 0.875rem;
            flex-shrink: 0;
            box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);
        }

        .message.user .message-avatar {
            background: linear-gradient(135deg, #3b82f6 0%, #1d4ed8 100%);
            color: white;
        }

        .message.assistant .message-avatar {
            background: linear-gradient(135deg, #0f172a 0%, #334155 100%);
            color: white;
        }

        .message-content {
            flex: 1;
            line-height: 1.6;
            padding: 0.75rem 0;
            color: #e2e8f0;
        }

        .message-content p {
            margin-bottom: 1rem;
        }

        .message-content p:last-child {
            margin-bottom: 0;
        }

        /* Markdown content styles */
        .message-content h1,
        .message-content h2,
        .message-content h3,
        .message-content h4,
        .message-content h5,
        .message-content h6 {
            color: #ececf1;
            margin: 1.5rem 0 1rem 0;
            font-weight: 600;
            line-height: 1.3;
        }

        .message-content h1 {
            font-size: 1.75rem;
            border-bottom: 2px solid #565869;
            padding-bottom: 0.5rem;
        }

        .message-content h2 {
            font-size: 1.5rem;
            border-bottom: 1px solid #565869;
            padding-bottom: 0.25rem;
        }

        .message-content h3 {
            font-size: 1.25rem;
        }

        .message-content h4 {
            font-size: 1.125rem;
        }

        .message-content h5,
        .message-content h6 {
            font-size: 1rem;
        }

        .message-content ul,
        .message-content ol {
            margin: 1rem 0;
            padding-left: 2rem;
        }

        .message-content li {
            margin: 0.5rem 0;
            line-height: 1.6;
        }

        .message-content blockquote {
            border-left: 4px solid #565869;
            padding-left: 1rem;
            margin: 1rem 0;
            font-style: italic;
            color: #b3b3b3;
        }

        .message-content code {
            background-color: #2d2d2d;
            color: #f8f8f2;
            padding: 0.2rem 0.4rem;
            border-radius: 0.25rem;
            font-family: 'SF Mono', 'Monaco', 'Inconsolata', 'Roboto Mono', 'Source Code Pro', monospace;
            font-size: 0.875rem;
        }

        .message-content pre {
            background-color: #2d2d2d;
            color: #f8f8f2;
            padding: 1rem;
            border-radius: 0.5rem;
            overflow-x: auto;
            margin: 1rem 0;
            border: 1px solid #565869;
        }

        .message-content pre code {
            background: none;
            padding: 0;
            border-radius: 0;
            font-size: 0.875rem;
        }

        .message-content table {
            width: 100%;
            border-collapse: collapse;
            margin: 1rem 0;
            border: 1px solid #565869;
        }

        .message-content th,
        .message-content td {
            border: 1px solid #565869;
            padding: 0.75rem;
            text-align: left;
        }

        .message-content th {
            background-color: #40414f;
        }

        .message-content tr:nth-child(even) {
            background-color: #40414f;
        }

        .message-content hr {
            border: none;
            border-top: 2px solid #565869;
            margin: 2rem 0;
        }

        .message-content a {
            color: #60a5fa;
            text-decoration: none;
        }

        .message-content a:hover {
            color: #93c5fd;
            text-decoration: underline;
        }

        .message-content strong {
            font-weight: 600;
            color: #ececf1;
        }

        .message-content em {
            font-style: italic;
            color: #d1d5db;
        }

        .message-time {
            font-size: 0.75rem;
            color: #8e8ea0;
            margin-top: 0.5rem;
        }

        .input-container {
            padding: 2rem;
            background: transparent;
            flex-shrink: 0;
            position: sticky;
            bottom: 0;
            z-index: 10;
        }

        .input-wrapper {
            display: flex;
            align-items: center;
            gap: 1rem;
            max-width: 100%;
            margin: 0 auto;
            background: linear-gradient(135deg, rgba(15, 23, 42, 0.9) 0%, rgba(30, 41, 59, 0.95) 100%);
            border: 1px solid rgba(255, 255, 255, 0.2);
            border-radius: 1rem;
            padding: 0.5rem 1rem;
            box-shadow: 0 8px 32px rgba(0, 0, 0, 0.4);
            backdrop-filter: blur(20px);
        }

        .message-input {
            flex: 1;
            background: transparent;
            border: none;
            outline: none;
            color: #e2e8f0;
            font-size: 1rem;
            resize: none;
            max-height: 200px;
            min-height: 48px;
            line-height: 1.6;
            font-family: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Oxygen, Ubuntu, Cantarell, 'Open Sans', 'Helvetica Neue', sans-serif;
            padding: 12px 0;
            display: flex;
            align-items: center;
        }

        .message-input::placeholder {
            color: #64748b;
            font-family: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Oxygen, Ubuntu, Cantarell, 'Open Sans', 'Helvetica Neue', sans-serif;
            font-weight: 300;
            letter-spacing: 0.02em;
            font-size: 0.95rem;
        }

        .send-button {
            background: linear-gradient(135deg, #3b82f6 0%, #1d4ed8 100%);
            border: none;
            border-radius: 0.75rem;
            color: white;
            cursor: pointer;
            padding: 0.75rem;
            display: flex;
            align-items: center;
            justify-content: center;
            transition: all 0.3s ease;
            box-shadow: 0 4px 15px rgba(59, 130, 246, 0.4);
        }

        .send-button:hover:not(:disabled) {
            background: linear-gradient(135deg, #2563eb 0%, #1e40af 100%);
            transform: translateY(-1px);
            box-shadow: 0 8px 25px rgba(59, 130, 246, 0.6);
        }

        .send-button:disabled {
            background: #94a3b8;
            cursor: not-allowed;
            transform: none;
            box-shadow: none;
        }

        .typing-indicator {
            display: none;
            align-items: center;
            gap: 0.5rem;
            color: #8e8ea0;
            font-style: italic;
            margin-bottom: 1rem;
            flex-shrink: 0;
        }

        .typing-dots {
            display: flex;
            gap: 0.25rem;
        }

.feedback-wrapper {
    position: right;
    display: right;
    flex-direction: column;
    align-items: flex-end;
    margin-top: 8px;
}

.feedback-actions {
    display: right;
    gap: 8px;
    align-items: center;
    justify-content: flex-end;
    position: right;
}

.ai-followup-heading {
            font-weight: 400;
            color: #ebebed;
            padding: 0.75rem 0;
            user-select: none;

            background: linear-gradient(135deg, rgba(59, 130, 246, 0.1) 0%, rgba(29, 78, 216, 0.05) 100%);
            border: 1px solid rgba(59, 130, 246, 0.3);
            border-radius: 0.75rem;
            margin-bottom: 1rem;
            font-size: 1.8rem;
            overflow: hidden;
            backdrop-filter: blur(10px);
        }

.feedback-button {
            cursor: pointer;
            font-weight: 600;
            color: #60a5fa;
            padding: 0.75rem 0;
            user-select: none;

            background: linear-gradient(135deg, rgba(59, 130, 246, 0.1) 0%, rgba(29, 78, 216, 0.05) 100%);
            border: 1px solid rgba(59, 130, 246, 0.3);
            border-radius: 0.75rem;
            margin-bottom: 1rem;
            font-size: 2.0rem;
            overflow: hidden;
            backdrop-filter: blur(10px);
        }

.feedback-button:hover {
    background-color: #2563eb; /* Slightly darker on hover */
}

.feedback-button-1 {
            cursor: pointer;
            font-weight: 600;
            color: #60a5fa;
            padding: 0.75rem 0;
            user-select: none;

            background: linear-gradient(135deg, rgba(59, 130, 246, 0.1) 0%, rgba(29, 78, 216, 0.05) 100%);
            border: 1px solid rgba(59, 130, 246, 0.3);
            border-radius: 0.75rem;
            margin-bottom: 1rem;
            font-size: 2.0rem;
            overflow: hidden;
            backdrop-filter: blur(10px);
        }

.feedback-button-1:hover {
    background-color: #2563eb; /* Slightly darker on hover */
}

.thumb-icon {
    font-size: 18px;
    color: #666;
}

.thumb-icon:hover {
    color: #222;
}

.three-dots {
    font-size: 20px;
    color: #888;
    cursor: pointer;
    background: none;
    border: none;
    padding: 0 4px;
}

.three-dots:hover {
    color: #000;
}

.feedback-popup {
    position: absolute;
    top: 100%; /* Directly below the toggle */
    right: 0;
    background: #f9f9f9;
    padding: 8px;
    border-radius: 6px;
    box-shadow: 0 2px 6px rgba(0,0,0,0.15);
    z-index: 10;
    margin-top: 4px;
}

.feedback-popup.hidden {
    display: none;
}

.feedback-text {
    width: 220px;
    padding: 6px;
    margin-right: 8px;
    border: 1px solid #ccc;
    border-radius: 4px;
}

.submit-button {
    padding: 6px 12px;
    background-color: #0078d4;
    color: white;
    border: none;
    border-radius: 4px;
    cursor: pointer;
}

.submit-button:hover {
    background-color: #005fa3;
}

        .typing-dot {
            width: 4px;
            height: 4px;
            border-radius: 50%;
            background-color: #8e8ea0;
            animation: typing 1.4s infinite;
        }

        .typing-dot:nth-child(2) {
            animation-delay: 0.2s;
        }

        .typing-dot:nth-child(3) {
            animation-delay: 0.4s;
        }

        @keyframes typing {
            0%, 60%, 100% {
                opacity: 0.3;
            }
            30% {
                opacity: 1;
            }
        }

        .error-message {
            background-color: #f56565;
            color: white;
            padding: 1rem;
            border-radius: 0.5rem;
            margin: 1rem;
            display: none;
        }

        .loading-spinner {
            display: inline-block;
            width: 16px;
            height: 16px;
            border: 2px solid #ececf1;
            border-radius: 50%;
            border-top-color: transparent;
            animation: spin 1s linear infinite;
        }

        @keyframes spin {
            to { transform: rotate(360deg); }
        }

        .messages-container {
            /* Modern browsers: thin auto-hiding scrollbar */
            scrollbar-width: thin;
            scrollbar-color: #565869 transparent;
        }

        /* Webkit browsers (Safari/Chrome): custom auto-hiding scrollbar */
        .messages-container::-webkit-scrollbar {
            width: 6px;
        }

        .messages-container::-webkit-scrollbar-track {
            background: transparent;
        }

        .messages-container::-webkit-scrollbar-thumb {
            background-color: #565869;
            border-radius: 3px;
            /* Auto-hide when not hovering/scrolling */
            opacity: 0;
            transition: opacity 0.3s ease;
        }

        .messages-container::-webkit-scrollbar-thumb:hover {
            background-color: #6f7089;
            opacity: 1;
        }

        /* Show scrollbar when container is hovered or being scrolled */
        .messages-container:hover::-webkit-scrollbar-thumb {
            opacity: 0.7;
        }

        .messages-container::-webkit-scrollbar-thumb:active {
            opacity: 1;
        }

        /* Execution round container for chronological layout */
        .execution-round-container {
            margin-bottom: 1.5rem;
        }

        .execution-round-container:last-child {
            margin-bottom: 0;
        }

        /* Round response content styling */
        .round-response-content {
            padding: 1rem;
            background: linear-gradient(135deg, rgba(15, 23, 42, 0.6) 0%, rgba(30, 41, 59, 0.7) 100%);
            border: 1px solid rgba(255, 255, 255, 0.1);
            border-radius: 0.75rem;
            backdrop-filter: blur(10px);
            color: #e2e8f0;
            line-height: 1.6;
            font-size: 0.875rem; /* Match sidebar font size */
        }

        .round-response-content p {
            margin-bottom: 1rem;
        }

        .round-response-content p:last-child {
            margin-bottom: 0;
        }

        /* Styles for the reasoning/details box */
        .reasoning-box {
            background: linear-gradient(135deg, rgba(59, 130, 246, 0.1) 0%, rgba(29, 78, 216, 0.05) 100%);
            border: 1px solid rgba(59, 130, 246, 0.3);
            border-radius: 0.75rem;
            margin-bottom: 1rem;
            font-size: 0.875rem;
            overflow: hidden;
            backdrop-filter: blur(10px);
        }



        .reasoning-box details {
            padding: 0.5rem 1rem;
        }

        .reasoning-box summary {
            cursor: pointer;
            font-weight: 600;
            color: #60a5fa;
            padding: 0.75rem 0;
            user-select: none;
            font-size: 0.875rem; /* Match tool box header size */
        }

        .reasoning-box summary:hover {
            color: #93c5fd;
        }

        .reasoning-content {
            padding-top: 0.75rem;
            border-top: 1px solid rgba(59, 130, 246, 0.3);
            margin-top: 0.5rem;
            color: #94a3b8; /* More greyish */
            line-height: 1.5;
            font-size: 0.8125rem; /* Smaller than main content */
        }

        .reasoning-content p {
            margin-bottom: 0.5rem;
        }

        .reasoning-content pre {
            white-space: pre-wrap;
            word-wrap: break-word;
            background-color: rgba(0, 0, 0, 0.3);
            padding: 0.75rem;
            border-radius: 0.375rem;
            font-family: 'Monaco', 'Menlo', 'Ubuntu Mono', monospace;
            font-size: 0.75rem; /* Even smaller for reasoning code */
            margin: 0.5rem 0;
        }

        .reasoning-content code {
            background-color: rgba(0, 0, 0, 0.3);
            padding: 0.2rem 0.4rem;
            border-radius: 0.25rem;
            font-family: 'Monaco', 'Menlo', 'Ubuntu Mono', monospace;
            font-size: 0.75rem; /* Even smaller for reasoning inline code */
        }

        /* Styles for the tool execution box */
        .tool-box {
            background: linear-gradient(135deg, rgba(139, 92, 246, 0.1) 0%, rgba(109, 40, 217, 0.05) 100%);
            border: 1px solid rgba(139, 92, 246, 0.3);
            border-radius: 0.75rem;
            margin-bottom: 1rem;
            font-size: 0.875rem;
            overflow: hidden;
            backdrop-filter: blur(10px);
        }

        .tool-box details {
            padding: 0.5rem 1rem;
        }

        .tool-box summary {
            cursor: pointer;
            font-weight: 600;
            color: #a78bfa;
            padding: 0.75rem 0;
            user-select: none;
            font-size: 0.875rem; /* Match reasoning box header size */
        }

        .tool-box summary:hover {
            color: #c4b5fd;
        }

        .tool-content {
            padding-top: 0.75rem;
            border-top: 1px solid rgba(139, 92, 246, 0.3);
            margin-top: 0.5rem;
            color: #94a3b8; /* More greyish like reasoning */
            line-height: 1.5;
            font-size: 0.8125rem; /* Smaller like reasoning */
        }

        .tool-content p {
            margin-bottom: 0.5rem;
            color: #94a3b8; /* More greyish like reasoning */
            font-size: 0.8125rem; /* Smaller like reasoning */
        }

        .tool-content pre {
            white-space: pre-wrap;
            word-wrap: break-word;
            background-color: rgba(0, 0, 0, 0.4);
            padding: 0.75rem;
            border-radius: 0.5rem;
            font-family: 'Monaco', 'Menlo', 'Ubuntu Mono', monospace;
            font-size: 0.75rem; /* Even smaller for tool code */
            margin: 0.5rem 0;
            color: #94a3b8; /* More greyish like reasoning */
            border: 1px solid rgba(139, 92, 246, 0.2);
        }

        .tool-content code {
            background-color: rgba(0, 0, 0, 0.4);
            padding: 0.2rem 0.4rem;
            border-radius: 0.25rem;
            font-family: 'Monaco', 'Menlo', 'Ubuntu Mono', monospace;
            font-size: 0.75rem; /* Even smaller for tool code */
            color: #94a3b8; /* More greyish like reasoning */
        }

        /* Show More button styling */
        .tool-content button:hover {
            background: rgba(139, 92, 246, 0.3) !important;
            border-color: rgba(139, 92, 246, 0.6) !important;
            color: #c4b5fd !important;
        }


.menu-button {
    position: absolute;
    bottom: 10px;
    right: 10px;
    background: none;
    border: none;
    font-size: 18px;
    color: white;
    cursor: pointer;
}


.menu-dropdown {
    position: absolute;
    right: 0;
    background: white;
    border: 1px solid #ccc;
    padding: 5px;
    z-index: 10;

}

.menu-dropdown.hidden {
    display: none;
}

.delete-button {
    position: absolute;
    bottom: 10px;
    right: 10px;
    background: none;
    border: none;
    cursor: pointer;
    padding: 0;
}

.delete-button svg {
    pointer-events: none;
}

/* Changelog overlay backdrop - blurs and darkens background */
.changelog-overlay {
    position: fixed;
    top: 0;
    left: 0;
    width: 100%;
    height: 100%;
    background: rgba(0, 0, 0, 0.5);
    backdrop-filter: blur(8px);
    -webkit-backdrop-filter: blur(8px);
    z-index: 999;
    display: none;
}

.changelog-overlay.visible {
    display: block;
}

.changelog-popup {
    position: fixed;
    top: 50%;
    left: 50%;
    transform: translate(-50%, -50%);
    z-index: 1000;
    background-color: rgba(30, 41, 59, 0.98);
    backdrop-filter: blur(20px);
    -webkit-backdrop-filter: blur(20px);
    color: #e2e8f0;
    padding: 30px;
    border-radius: 16px;
    box-shadow: 0 25px 50px rgba(0, 0, 0, 0.6), 0 0 0 1px rgba(255, 255, 255, 0.1);
    width: 90%;
    max-width: 600px;
    max-height: 80vh;
    display: none;
    flex-direction: column;
}

.changelog-popup.visible {
    display: flex;
}

.changelog-popup h2 {
    margin-bottom: 20px;
    font-size: 1.5rem;
    font-weight: 600;
    text-align: center;
    background: linear-gradient(135deg, #60a5fa, #3b82f6);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
}

.changelog-popup #changelog-content {
    max-height: 400px;
    overflow-y: auto;
    margin-bottom: 20px;
    scrollbar-width: thin;
    scrollbar-color: #565869 transparent;
    text-align: left;
    padding: 0 15px;
}

/* Markdown content styling inside changelog */
.changelog-popup #changelog-content h1,
.changelog-popup #changelog-content h2,
.changelog-popup #changelog-content h3 {
    text-align: left;
    margin-top: 1rem;
    margin-bottom: 0.5rem;
    color: #e2e8f0;
}

.changelog-popup #changelog-content h1 {
    font-size: 1.25rem;
    border-bottom: 1px solid rgba(255, 255, 255, 0.1);
    padding-bottom: 0.5rem;
}

.changelog-popup #changelog-content h2 {
    font-size: 1.1rem;
    color: #60a5fa;
}

.changelog-popup #changelog-content h3 {
    font-size: 1rem;
    color: #94a3b8;
}

.changelog-popup #changelog-content ul,
.changelog-popup #changelog-content ol {
    text-align: left;
    padding-left: 1.5rem;
    margin: 0.5rem 0;
}

.changelog-popup #changelog-content li {
    text-align: left;
    margin: 0.25rem 0;
    line-height: 1.5;
}

.changelog-popup #changelog-content p {
    text-align: left;
    margin: 0.5rem 0;
}

.changelog-popup #changelog-content a {
    color: #93c5fd;
    text-decoration: underline;
}

.changelog-popup #changelog-content a:visited {
    color: #b0c8ff; /* keep contrast; avoid “muddy” visited purple */
}

.changelog-popup #changelog-content a:hover {
    color: #bfdbfe;
    text-decoration: underline;
}

.changelog-popup #changelog-content::-webkit-scrollbar {
    width: 6px;
}

.changelog-popup #changelog-content::-webkit-scrollbar-thumb {
    background-color: #565869;
    border-radius: 3px;
}

.changelog-popup button {
    background: linear-gradient(135deg, #3b82f6 0%, #1d4ed8 100%);
    color: white;
    border: none;
    padding: 10px 20px;
    border-radius: 8px;
    font-size: 1rem;
    font-weight: 600;
    cursor: pointer;
    transition: all 0.3s ease;
    box-shadow: 0 4px 15px rgba(59, 130, 246, 0.4);
    align-self: center;
}

.changelog-popup button:hover {
    background: linear-gradient(135deg, #2563eb 0%, #1e40af 100%);
    transform: translateY(-2px);
}

/* Changelog button highlight animation for new updates */
@keyframes changelog-pulse {
    0% {
        box-shadow: 0 0 0 0 rgba(59, 130, 246, 0.7);
    }
    50% {
        box-shadow: 0 0 0 8px rgba(59, 130, 246, 0);
    }
    100% {
        box-shadow: 0 0 0 0 rgba(59, 130, 246, 0);
    }
}

@keyframes changelog-glow {
    0%, 100% {
        background: rgba(59, 130, 246, 0.2);
        border-color: rgba(59, 130, 246, 0.5);
    }
    50% {
        background: rgba(59, 130, 246, 0.4);
        border-color: rgba(59, 130, 246, 0.8);
    }
}

.version-button.has-updates {
    animation: changelog-pulse 2s infinite, changelog-glow 2s infinite;
    border: 1px solid rgba(59, 130, 246, 0.5);
    border-radius: 0.5rem;
    color: #60a5fa;
}


    </style>
</head>
<body>
    <!-- Login Screen (temporarily disabled) -->
    <!--
    <div id="login-screen" class="login-container">
        <div class="login-card">
            <h1>Tribal Knowledge Assistant</h1>
            <p style="margin-bottom: 2rem; color: #cbd5e1;">Please sign in with your ASML account to continue</p>
            <button id="login-button" class="login-button">
                <svg width="20" height="20" viewBox="0 0 23 23" fill="none">
                    <path d="M11 11H0V0H11V11Z" fill="#F25022"/>
                    <path d="M23 11H12V0H23V11Z" fill="#7FBA00"/>
                    <path d="M11 23H0V12H11V23Z" fill="#00A4EF"/>
                    <path d="M23 23H12V12H23V23Z" fill="#FFB900"/>
                </svg>
                Sign in with Microsoft
            </button>
        </div>
    </div>
    -->

    <!-- Chat Interface (hidden by default) -->
    <div id="chat-interface" style="display: none;">
        <div class="header">
            <div class="header-left">
                <button id="sidebar-toggle" class="sidebar-toggle">
                    <div class="sidebar-icon"></div>
                </button>
                <h1>Tribal Knowledge Assistant</h1>
            </div>
            <!-- <div class="header-right">
                <div class="user-info">
                    <div class="user-avatar" id="user-avatar">U</div>
                    <span id="user-name">User</span>
                </div>
                <button id="logout-button" class="logout-button">Sign Out</button>
                <div class="status-indicator">
                    <div class="status-dot"></div>
                    <span id="status-text">Connected</span>
                    <button id="changelog-toggle" class="version-text" style="background: none; border: none; cursor: pointer;">v2.7.1</button>
                    <div id="changelog-popup" class="changelog-popup">
                        <h2>Recent Changes</h2>
                        <div id="changelog-content"></div>
                        <button id="close-changelog-button">OK</button>
                    </div>
                </div>
            </div> -->

            <div class="header-right ">
                <!-- Status (Connected/Thinking, Green Dot) -->
                <div class="status-indicator">
                    <span id="status-text">Connected</span>
                    <div class="status-dot"></div>
                </div>
                <!-- User Name -->
                <div class="user-info header-pill">
                    <div class="user-avatar" id="user-avatar">U</div>
                    <span id="user-name">User</span>
                </div>
                <!-- Version (clickable) -->
                <button id="changelog-toggle" class="version-button header-pill" title="View Changelog">
                v2.7.1</button>
                <!-- Logout icon -->
                <button id="logout-button" class="logout-button header-pill" title="Sign Out">
                <svg xmlns="http://www.w3.org/2000/svg" width="20" height="20" viewBox="0 0 24
                24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round"
                stroke-linejoin="round">
                <path d="M9 21H5a2 2 0 0 1-2-2 V5a2 2 0 0 1 2-2h4"/>
                <polyline points="16 17 21 12 16 7"/>
                <line x1="21" y1="12" x2="9" y2=" 12"/>
                </svg>
                </button>
                </div>
        </div>

        <!-- Changelog overlay and popup - moved outside header for proper stacking -->
        <div id="changelog-overlay" class="changelog-overlay"></div>
        <div id="changelog-popup" class="changelog-popup">
            <h2>What's New</h2>
            <div id="changelog-content"></div>
            <button id="close-changelog-button">Got it!</button>
        </div>

        <div class="main-layout">
            <!-- Sidebar -->
            <div class="sidebar" id="sidebar">
                <div class="sidebar-content">
                    <button id="new-chat-button" class="sidebar-new-chat-button">
                        <svg width="16" height="16" viewBox="0 0 24 24" fill="currentColor">
                            <path d="M12 2C6.48 2 2 6.48 2 12s4.48 10 10 10 10-4.48 10-10S17.52 2 12 2zm5 11h-4v4h-2v-4H7v-2h4V7h2v4h4v2z"/>
                        </svg>
                        New conversation
                    </button>

                    <div class="chat-history">
                        <div class="chat-history-header">
                            <h3>Recent Chats</h3>
                        </div>
                        <div class="chat-list" id="chat-list"></div>
                    </div>
                </div>
            </div>

            <!-- Main Content Area -->
            <div class="main-content">
                <div class="error-message" id="error-message"></div>

                <div class="chat-container">
                <div class="messages-container" id="messages-container">
                <!-- Messages will be added here dynamically -->
            </div>

            <div class="typing-indicator" id="typing-indicator">
                <div class="message-avatar">
                    <span>AI</span>
                </div>
                <div>
                    <span>AI is typing</span>
                    <div class="typing-dots">
                        <div class="typing-dot"></div>
                        <div class="typing-dot"></div>
                        <div class="typing-dot"></div>
                    </div>
                </div>
            </div>

            <div class="input-container">
                <div class="input-wrapper">
                    <textarea
                        id="message-input"
                        class="message-input"
                        placeholder="Send a message..."
                        rows="1"
                    ></textarea>
                    <button id="send-button" class="send-button">
                        <svg width="20" height="20" viewBox="0 0 24 24" fill="currentColor">
                            <path d="M2.01 21L23 12 2.01 3 2 10l15 2-15 2z"/>
                        </svg>
                    </button>
                </div>
            </div>
            </div> <!-- Close main-content -->
        </div> <!-- Close main-layout -->
    </div>

    <script>
        // Azure Entra ID Configuration - will be injected by the server
        const msalConfig = {
            auth: {
                clientId: '{{ CLIENT_ID }}',
                authority: 'https://login.microsoftonline.com/{{ TENANT_ID }}',
                redirectUri: window.location.origin
            },
            cache: {
                cacheLocation: 'sessionStorage',
                storeAuthStateInCookie: false
            }
        };

        // MSAL instance
        const msalInstance = new msal.PublicClientApplication(msalConfig);

        // Request scopes - using the standard user_impersonation scope
        const loginRequest = {
            scopes: [`api://${msalConfig.auth.clientId}/user_impersonation`]
        };

        let accessToken = null;

        // Configure marked for better security and formatting
        marked.setOptions({
            breaks: true,
            gfm: true,
            sanitize: false,
            smartLists: true,
            smartypants: true,
            highlight: function(code, lang) {
                // Basic syntax highlighting could be added here if needed
                return code;
            }
        });